# Build synthetic component maps (Planck 6-channel, Nside=4096)

Populate `/rds/rds-lxu/flamingo/integrated_maps_synthetic/components/` with
beam-**unconvolved** maps at the six Planck HFI frequencies from
`reference_tables/planck_info.png` (Table I):

| GHz | 100 | 143 | 217 | 353 | 545 | 857 |
|-----|-----|-----|-----|-----|-----|-----|

**Components are stored separately — not summed.** Beam convolution,
coaddition, and noise will be applied in a later step.

Each component is in $\mu\mathrm{K}_\mathrm{CMB}$ (or Compton $y$ / Doppler
$b$ / Jy/sr for the native FLAMINGO products) at native
$N_\mathrm{side}=4096$, beam-unconvolved.

| Component | Source | Action |
|-----------|--------|--------|
| CMB | FLAMINGO $\kappa$ + CAMB $C_\ell^{TT}$ | simulate (pixell lensing) |
| CIB | Yang26 released bands 217/353/545/857 | copy intensity FITS; 100/143 GHz via greybody SED |
| tSZ | `lensed_tSZ_rot.fits` (Compton $y$) | copy $y$; build $\Delta T(\nu)=T_\mathrm{CMB}\,y\,f(\nu)$ |
| kSZ | `lensed_kSZ_rot.fits` (Doppler $b$) | copy $b$; $\Delta T=-T_\mathrm{CMB}\,b$ (freq.-indep.) |

Input maps: `/rds/flamingo/L2800N5040/HYDRO_FIDUCIAL/lightcone0_shells`.

Requires `pip install -e ".[cmb]"` for the lensed CMB step (`camb`, `pixell`).

In [ ]:
from pathlib import Path

from flamingo_mock import MockConfig
from flamingo_mock.config import PLANCK_FREQUENCIES_GHZ
from flamingo_mock import cib, cmb, ksz, tsz

cfg = MockConfig(
    frequencies=PLANCK_FREQUENCIES_GHZ,
    nside=4096,
    seed=42,
)
cfg.make_dirs()

OUT = cfg.out_dir / "components"
for sub in ("cmb", "cib", "tsz", "ksz"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)

print("data:", cfg.data_dir)
print("out: ", OUT)
print("freqs:", list(cfg.frequencies))
print("Nside:", cfg.nside)

## 1. Lensed primary CMB

In [ ]:
# ~30–60 min at Nside=4096; skipped if the FITS already exists.
cmb_uK = cmb.make_lensed_cmb(cfg, out_dir=OUT / "cmb")
print(f"CMB lensed: std={cmb_uK.std():.2f} uK")

## 2. CIB — copy released bands, approximate 100/143 GHz

Released lensed bandpass maps (217/353/545/857 GHz) are copied from the
FLAMINGO tree. **100 and 143 GHz** are outside the released set; we build
them with the three-parameter greybody SED at $z_\mathrm{eff}=1.5$ (same
method as `flamingo_mock.cib.approximate_cib_intensity`).

Note: `CIB_nonrot_BANDPASS_F143_three_params.fits` exists but is **not**
lensed — we do not use it.

In [ ]:
# Archive released intensity maps [Jy/sr] (symlink to save space)
cib.copy_released_cib_intensity(cfg, out_dir=OUT / "cib", use_symlink=True)

# Thermodynamic maps [uK_CMB] at all six frequencies
cib_uK = cib.make_cib_maps(cfg, out_dir=OUT / "cib")

## 3. tSZ — copy Compton-$y$, build $\Delta T(\nu)$

The lensed Compton-$y$ map is archived from `lensed_tSZ_rot.fits`. Per-frequency
temperature maps use the non-relativistic spectral function
$f(x)=x\coth(x/2)-4$.

In [ ]:
tsz.archive_compton_y(cfg, out_dir=OUT / "tsz", use_symlink=True)
tsz_uK = tsz.make_tsz_maps(cfg, out_dir=OUT / "tsz")

## 4. kSZ — copy Doppler-$b$, convert to $\mu\mathrm{K}_\mathrm{CMB}$

The lensed kSZ map (`lensed_kSZ_rot.fits`) stores Doppler $b$ with
$\Delta T/T_\mathrm{CMB}=-b$. The thermodynamic map is frequency independent.

In [ ]:
ksz.archive_doppler_b(cfg, out_dir=OUT / "ksz", use_symlink=True)
ksz_uK = ksz.make_ksz_map(cfg, out_dir=OUT / "ksz")
print(f"kSZ dT: std={ksz_uK.std():.3e} uK")

## 5. Inventory

In [ ]:
print(f"\nProducts under {OUT}:\n")
for sub in sorted(OUT.iterdir()):
    if sub.is_dir():
        files = sorted(sub.glob("*.fits"))
        print(f"  {sub.name}/  ({len(files)} files)")
        for p in files:
            print(f"    {p.name}  ({p.stat().st_size/1e9:.2f} GB)")

## 6. Publication-quality Mollweide maps

Full-resolution products remain at $N_\mathrm{side}=4096$ on disk. For
visualisation we **downgrade to `NSIDE_VIZ`** with `hp.ud_grade` (fast
interactive rendering; does not alter stored maps).

Figures are saved to `../figures/` as PDF + PNG.

In [ ]:
import healpy as hp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

NSIDE_VIZ = 256  # preview resolution for mollview only

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "legend.frameon": False,
})


def savefig(fig, name: str) -> None:
    fig.savefig(FIG_DIR / f"{name}.pdf")
    fig.savefig(FIG_DIR / f"{name}.png")
    print(f"Wrote {FIG_DIR / name}.pdf")


def load_for_viz(path: Path) -> np.ndarray:
    m = hp.read_map(str(path), dtype=np.float64, )
    if hp.get_nside(m) != NSIDE_VIZ:
        m = hp.ud_grade(m, NSIDE_VIZ)
    return m


def mollview_pub(m, title, unit, *, sub=None, fig=None, symmetric=True,
                 pct=(1, 99), cmap="RdBu_r", log_scale=False):
    # Do not call plt.tight_layout() on multi-panel healpy figures.
    x = np.asarray(m, dtype=np.float64)
    kw = dict(title=title, unit=unit, cmap=cmap, hold=True, notext=False, xsize=900)
    if sub is not None:
        kw["sub"] = sub
    if fig is not None:
        kw["fig"] = fig
    if log_scale:
        x = np.where(x > 0, x, np.nan)
        vmin, vmax = np.nanpercentile(x, pct)
        hp.mollview(x, norm="log", min=vmin, max=vmax, **kw)
    elif symmetric:
        vmax = np.percentile(np.abs(x[np.isfinite(x)]), pct[1])
        hp.mollview(x, min=-vmax, max=vmax, **kw)
    else:
        vmin, vmax = np.percentile(x[np.isfinite(x)], pct)
        hp.mollview(x, min=vmin, max=vmax, **kw)
    hp.graticule(dpar=30, dmer=60, alpha=0.35)

print(f"viz Nside={NSIDE_VIZ}, fig dir={FIG_DIR.resolve()}")

### 6.1 Lensed primary CMB

In [ ]:
cmb_path = OUT / "cmb" / f"primary_CMB_T_lensed_nside{cfg.nside}_seed{cfg.seed}.fits"
m_cmb = load_for_viz(cmb_path)

fig = plt.figure(figsize=(7.5, 4.5))
mollview_pub(
    m_cmb,
    title=rf"Lensed primary CMB ($N_{{\mathrm{{viz}}}}={NSIDE_VIZ}$)",
    unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
    fig=fig,
)
savefig(fig, "components_cmb_lensed_mollview")
plt.show()

### 6.2 tSZ — Compton $y$ and $\Delta T(\nu)$

Compton-$y$ is dimensionless; per-frequency panels use symmetric scales set
by the 99th percentile of $|\Delta T|$ at each band.

In [ ]:
y_path = OUT / "tsz" / f"compton_y_nside{cfg.nside}.fits"
m_y = load_for_viz(y_path)

fig = plt.figure(figsize=(7.5, 4.5))
mollview_pub(
    m_y,
    title=r"Compton $y$ (lensed tSZ)",
    unit="$y$",
    fig=fig,
    symmetric=True,
    pct=(0.5, 99.5),
)
savefig(fig, "components_tsz_compton_y_mollview")
plt.show()

freqs = list(cfg.frequencies)
ncols, nrows = 3, 2
fig = plt.figure(figsize=(4.8 * ncols, 3.8 * nrows))
for i, nu in enumerate(freqs, start=1):
    p = OUT / "tsz" / f"tSZ_deltaT_{nu:.0f}GHz_nside{cfg.nside}.fits"
    mollview_pub(
        load_for_viz(p),
        title=rf"tSZ $\Delta T$ — ${nu:.0f}\,\mathrm{{GHz}}$",
        unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
        sub=(nrows, ncols, i),
        fig=fig,
        symmetric=True,
    )
fig.suptitle(
    rf"Thermal SZ temperature maps (beam-unconvolved, $N_{{\mathrm{{viz}}}}={NSIDE_VIZ}$)",
    y=1.02, fontsize=13,
)
savefig(fig, "components_tsz_deltaT_allfreq_mollview")
plt.show()

### 6.3 kSZ — Doppler $b$ and thermodynamic map

In [ ]:
fig = plt.figure(figsize=(12.0, 4.2))
mollview_pub(
    load_for_viz(OUT / "ksz" / f"doppler_b_nside{cfg.nside}.fits"),
    title=r"Doppler $b$ (lensed kSZ)",
    unit="$b$",
    sub=(1, 2, 1), fig=fig, symmetric=True,
)
mollview_pub(
    load_for_viz(OUT / "ksz" / f"kSZ_deltaT_nside{cfg.nside}.fits"),
    title=r"kSZ $\Delta T$ (all frequencies)",
    unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
    sub=(1, 2, 2), fig=fig, symmetric=True,
)
fig.suptitle(rf"Kinetic SZ ($N_{{\mathrm{{viz}}}}={NSIDE_VIZ}$)", y=1.02, fontsize=13)
savefig(fig, "components_ksz_mollview")
plt.show()

### 6.4 CIB — $\Delta T(\nu)$ at six Planck bands

CIB is positive-definite on average; colour limits use the 1st–99th
percentile at each frequency.

In [ ]:
freqs = list(cfg.frequencies)
ncols, nrows = 3, 2
fig = plt.figure(figsize=(4.8 * ncols, 3.8 * nrows))
for i, nu in enumerate(freqs, start=1):
    p = OUT / "cib" / f"CIB_deltaT_{nu:.0f}GHz_nside{cfg.nside}.fits"
    approx = "" if nu in (217, 353, 545, 857) else " (SED approx.)"
    mollview_pub(
        load_for_viz(p),
        title=rf"CIB $\Delta T$ — ${nu:.0f}\,\mathrm{{GHz}}${approx}",
        unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
        sub=(nrows, ncols, i), fig=fig,
        symmetric=False, pct=(1, 99), cmap="viridis",
    )
fig.suptitle(
    rf"CIB temperature maps (beam-unconvolved, $N_{{\mathrm{{viz}}}}={NSIDE_VIZ}$)",
    y=1.02, fontsize=13,
)
savefig(fig, "components_cib_deltaT_allfreq_mollview")
plt.show()

### 6.5 Overview — one panel per component (353 GHz)

In [ ]:
nu_ref = 353.0
fig = plt.figure(figsize=(14.0, 3.6))
panels = [
    (OUT / "cmb" / f"primary_CMB_T_lensed_nside{cfg.nside}_seed{cfg.seed}.fits",
     "CMB", r"$\mu\mathrm{K}_\mathrm{CMB}$", True, "RdBu_r"),
    (OUT / "tsz" / f"tSZ_deltaT_{nu_ref:.0f}GHz_nside{cfg.nside}.fits",
     rf"tSZ ${nu_ref:.0f}\,\mathrm{{GHz}}$", r"$\mu\mathrm{K}_\mathrm{CMB}$", True, "RdBu_r"),
    (OUT / "ksz" / f"kSZ_deltaT_nside{cfg.nside}.fits",
     "kSZ", r"$\mu\mathrm{K}_\mathrm{CMB}$", True, "RdBu_r"),
    (OUT / "cib" / f"CIB_deltaT_{nu_ref:.0f}GHz_nside{cfg.nside}.fits",
     rf"CIB ${nu_ref:.0f}\,\mathrm{{GHz}}$", r"$\mu\mathrm{K}_\mathrm{CMB}$", False, "viridis"),
]
for i, (path, title, unit, sym, cmap) in enumerate(panels, start=1):
    mollview_pub(load_for_viz(path), title=title, unit=unit,
                 sub=(1, 4, i), fig=fig, symmetric=sym, cmap=cmap,
                 pct=(1, 99) if not sym else (1, 99))
fig.suptitle(
    rf"Component overview at ${nu_ref:.0f}\,\mathrm{{GHz}}$ "
    rf"($N_{{\mathrm{{viz}}}}={NSIDE_VIZ}$, beam-unconvolved)",
    y=1.05, fontsize=13,
)
savefig(fig, "components_overview_353GHz_mollview")
plt.show()